In [1]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
set_seed(42)

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.models.network import DiffusionSSSD
from src.models.gaussian_noise import GaussianDiffusion
from src.train.trainer import setup_optimizer, DiffusionTrainer

/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF


CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.
Falling back on slow Cauchy and Vandermonde kernel. Install at least one of pykeops or the CUDA extension for better speed and memory efficiency.
/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Global hyperpar
EPOCHS = 25
BATCH_SIZE = 64
LR = 0.0007
WEIGHT_DECAY = 0.07
TIMESTEPS = 150

TEST_INHIBITOR = "2-mercaptobenzimidazole" 

NUM_CYCLE = [1, 2, 3, 4]
save_dir = project_root / "experiments" / "run_01"
SAVE_DIR = str(save_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

[*] Device: cuda


In [3]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=TEST_INHIBITOR, 
    norm_feat=True, 
    use_wavelet=False
)

train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)

val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

Size Train: 2684 samples
Size Val: 776 samples


In [5]:
num_desc_features = train_dataset[0]["features"].shape[0]

net = DiffusionSSSD(
        in_channels=1, 
        desc_features=num_desc_features, 
        base_channels=32
    )
    
diffusion = GaussianDiffusion(model=net, timesteps=TIMESTEPS)

optimizer, scheduler = setup_optimizer(
    model=net, 
    lr=LR, 
    weight_decay=WEIGHT_DECAY, 
    epochs=EPOCHS
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Group 0: lr=0.0007, weight_decay=0.07, params=195553
Group 1: lr=0.0006, weight_decay=0.0, params=70400


In [6]:
trainer = DiffusionTrainer(
        diffusion_model=diffusion,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        save_dir=SAVE_DIR,
        vol_scaler=pipe.vol_scaler2,
        cur_scaler=pipe.cur_scaler2
    )

print("\n" + "="*40)
print("Start")
print("="*40)
trainer.fit(epochs=EPOCHS)


Start
Teaching on cuda...


Sampling: 100%|██████████| 150/150 [00:03<00:00, 37.86it/s]


Epoch 1 | Train Loss: 1.1833 | Val Loss: 2.2397 | LR: 0.000697 | MSE_loss 1.183330 | area_loss 0.000000 | ratio_loss 0.000000 | peaks_loss 0.000000
Saved best model (Val Loss: 2.2397)


Sampling: 100%|██████████| 150/150 [00:03<00:00, 43.05it/s]


Epoch 2 | Train Loss: 0.5556 | Val Loss: 1.8463 | LR: 0.000689 | MSE_loss 0.555568 | area_loss 0.000000 | ratio_loss 0.000000 | peaks_loss 0.000000
Saved best model (Val Loss: 1.8463)


...

Epoch 23 | Train Loss: 0.2397 | Val Loss: 0.4787 | LR: 0.000011 | MSE_loss 0.239660 | area_loss 0.000000 | ratio_loss 0.000000 | peaks_loss 0.000000


Sampling: 100%|██████████| 150/150 [00:04<00:00, 36.55it/s]


Epoch 24 | Train Loss: 0.2534 | Val Loss: 0.4899 | LR: 0.000003 | MSE_loss 0.253383 | area_loss 0.000000 | ratio_loss 0.000000 | peaks_loss 0.000000


Sampling: 100%|██████████| 150/150 [00:04<00:00, 33.98it/s]


Epoch 25 | Train Loss: 0.2303 | Val Loss: 0.4792 | LR: 0.000000 | MSE_loss 0.230336 | area_loss 0.000000 | ratio_loss 0.000000 | peaks_loss 0.000000


In [ ]:
def forward(self, x_start, descriptors):
    ...
    predicted_current = self.model(signal=x_noisy, descriptors=descriptors, t=t)

    # Взвешенный MSE (больше внимания пикам)
    weight = 1.0 + 4.0 * torch.abs(x_start)
    mse_loss = (weight * (predicted_current - x_start)**2).mean()

    # Штраф за выход за границы (лучше квадратичный)
    over = F.relu(torch.abs(predicted_current) - 1.0)
    loss_bounds = (over ** 2).mean()

    # Площади положительных и отрицательных частей
    mask_pos = (x_start > 0).float()
    mask_neg = (x_start < 0).float()
    area_pos_pred = (predicted_current * mask_pos).sum(dim=-1)
    area_pos_true = (x_start * mask_pos).sum(dim=-1)
    area_neg_pred = (-predicted_current * mask_neg).sum(dim=-1)
    area_neg_true = (-x_start * mask_neg).sum(dim=-1)
    loss_area = F.mse_loss(area_pos_pred, area_pos_true) + F.mse_loss(area_neg_pred, area_neg_true)

    # Веса (подбираются)
    total_loss = mse_loss + 0.5 * loss_bounds + 0.1 * loss_area
    return total_loss, mse_loss, loss_bounds, loss_area  # если нужно логировать